# Kaggle Master Pipeline

This notebook demonstrates how to cleanly execute the modular Python pipeline for both Cat Face Detection (YOLOv11 Object Detection) and Cat Prey Classification (YOLOv11 Classification). 
We have structured the logic into `src/face_detection` and `src/prey_detection` so you can easily switch between dataset configurations and run training right from this notebook.

## 1. Setup Environment

In [ ]:
%rm -r /kaggle/working/catflap%cd /kaggle/working!git clone https://github.com/clemenshog42/catflap.git
%cd catflap
!pip install -r requirements.txt
import os
import shutil

## 2. Train Cat Face Detector

First, we build the dataset. You can configure the `source` (`annotation` for `.cat` files, `model` for a pre-trained model) and the `color` (`rgb` or `grayscale`).

In [ ]:
FACE_INPUT_DIR = "/kaggle/input/catsyoloannotated/content/cat_dataset" # Adjust this to your dataset
FACE_OUTPUT_DIR = "/kaggle/working/face_dataset"

!python src/face_detection/dataset_builder.py \
    --input_dir {FACE_INPUT_DIR} \
    --output_dir {FACE_OUTPUT_DIR} \
    --source annotation \
    --color grayscale \
    --apply_clahe

In [ ]:
# Train the YOLO face detector on the generated dataset
!python src/face_detection/train.py \
    --data_yaml {FACE_OUTPUT_DIR}/dataset.yaml \
    --epochs 100 \
    --imgsz 640 \
    --project "yolov11_catmouth"

## 3. Train Cat Prey Classifier

Now we process the dataset for prey classification. We use the Face Model we just trained to automatically crop the cat faces before feeding them to the classifier.

### Optional: Test Cropping Padding
Before building the dataset, you can test how the bounding box padding looks on a few sample images.

In [ ]:
PREY_DIR = "/kaggle/input/cleanpreyfacesrgb/CleanPreyData_v1/prey"
TRAINED_FACE_MODEL = "/kaggle/working/yolov11_catmouth/train/weights/best.pt"

!python src/prey_detection/test_padding.py \
    --input_dir {PREY_DIR} \
    --output_dir /kaggle/working/padding_test \
    --crop_model_path {TRAINED_FACE_MODEL} \
    --pad_w 15 \
    --pad_top 10 \
    --pad_bottom 50 \
    --num_samples 5

print("Check /kaggle/working/padding_test to see the original bounding boxes (Red) vs the padded crops (Green)!")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

test_dir = Path('/kaggle/working/padding_test')
crop_images = list(test_dir.glob('*_crop.jpg'))
box_images = list(test_dir.glob('*_boxes.jpg'))

if crop_images and box_images:
    # Show up to 5 pairs
    num_to_show = min(5, len(crop_images))
    fig, axes = plt.subplots(num_to_show, 2, figsize=(10, 5 * num_to_show))
    if num_to_show == 1:
        axes = [axes]
    
    for i in range(num_to_show):
        # Find corresponding box and crop images
        # Assuming files are named test_X_crop.jpg and test_X_boxes.jpg
        box_img = mpimg.imread(str(box_images[i]))
        crop_img = mpimg.imread(str(str(box_images[i]).replace('_boxes.jpg', '_crop.jpg')))
        
        axes[i][0].imshow(box_img)
        axes[i][0].set_title("Original (Red) & Padded (Green)")
        axes[i][0].axis('off')
        
        axes[i][1].imshow(crop_img)
        axes[i][1].set_title("Final Padded Crop")
        axes[i][1].axis('off')
        
    plt.tight_layout()
    plt.show()
else:
    print("No test images found.")


In [ ]:
PREY_DIR = "/kaggle/input/cleanpreyfacesrgb/CleanPreyData_v1/prey" # Contains cats with prey
CLEAN_DIR = "/kaggle/input/cleanpreyfacesrgb/CleanPreyData_v1/clean" # Contains cats without prey
PREY_OUTPUT_DIR = "/kaggle/working/prey_dataset"
TRAINED_FACE_MODEL = "/kaggle/working/yolov11_catmouth/train/weights/best.pt" # Path to model from previous step

!python src/prey_detection/dataset_pipeline.py \
    --prey_dir {PREY_DIR} \
    --clean_dir {CLEAN_DIR} \
    --output_dir {PREY_OUTPUT_DIR} \
    --crop_model_path {TRAINED_FACE_MODEL} \
    --pad_w 10 \
    --pad_top 10 \
    --pad_bottom 50 \
    --color grayscale \
    --apply_clahe


In [ ]:
# Train the YOLO Classification Model
!python src/prey_detection/train.py \
    --data_dir {PREY_OUTPUT_DIR} \
    --epochs 50 \
    --imgsz 224 \
    --project "clean_prey_yolov11n_cls"